### Upload scraped articles to S3

**articles scraped in doc "scraping_project_new.ipynb"**

In [1]:
# import libraries
import os
import pprint
import random
import string
import boto3

In [2]:
# do I need the pretty printer?
# Initialize pretty printer for better output formatting
pp = pprint.PrettyPrinter(indent=2)

# Create S3 client using default credentials from AWS CLI
# boto3 will automatically use credentials from ~/.aws/credentials
s3 = boto3.client(
    "s3",
    region_name="eu-west-1")

In [3]:
# Let's first check what buckets already exist in your AWS account
# This helps us understand what resources we're starting with
print("📋 Listing all S3 buckets in your account...")
response = s3.list_buckets()

print("\n📦 Raw response from AWS:")
pp.pprint(response)

print("\n📦 Your current S3 buckets:")
if response["Buckets"]:
    for bucket in response["Buckets"]:
        print(f"- {bucket['Name']}")
else:
    print("No buckets found in your account")

print(f"\n✅ Successfully retrieved {len(response['Buckets'])} buckets")

📋 Listing all S3 buckets in your account...

📦 Raw response from AWS:
{ 'Buckets': [ { 'BucketArn': 'arn:aws:s3:::22508939',
                 'CreationDate': datetime.datetime(2025, 11, 30, 1, 17, 29, tzinfo=tzlocal()),
                 'Name': '22508939'},
               { 'BucketArn': 'arn:aws:s3:::2400414',
                 'CreationDate': datetime.datetime(2025, 12, 1, 15, 29, 53, tzinfo=tzlocal()),
                 'Name': '2400414'},
               { 'BucketArn': 'arn:aws:s3:::2400573',
                 'CreationDate': datetime.datetime(2025, 12, 2, 11, 40, 20, tzinfo=tzlocal()),
                 'Name': '2400573'},
               { 'BucketArn': 'arn:aws:s3:::2404422-news-sentiment',
                 'CreationDate': datetime.datetime(2025, 12, 16, 14, 46, 57, tzinfo=tzlocal()),
                 'Name': '2404422-news-sentiment'},
               { 'BucketArn': 'arn:aws:s3:::2405799-homework',
                 'CreationDate': datetime.datetime(2025, 12, 1, 22, 5, 40, tzinfo=tzlocal(

In [ ]:
# Now we'll create a function to generate unique bucket names
# S3 bucket names must be globally unique across all AWS accounts
print("🔧 Setting up bucket name generator...")


def generate_bucket_name(base_name):
    """Generate a unique bucket name using base name and random digits"""
    random_part = "".join(random.choices(string.digits, k=3))
    return f"{base_name}-{random_part}"


# Generate a unique bucket name - replace 'add-your-name-here' with your name!
my_name = "add-your-name-here"  # TODO: Change this!
bucket_name = generate_bucket_name(my_name)

if my_name == "add-your-name-here":
    print("❌ Remember to change 'add-your-name-here' to your actual name!")
else:
    print("✅ Name generator ready")
    print(f"📝 Your generated bucket name: {bucket_name}")

In [ ]:
# Create a new S3 bucket with our generated name
# We'll specify EU (Ireland) as our region
default_region = "eu-west-1"
print(f"🚀 Creating new bucket: {bucket_name}")
print(f"🌍 Region: {default_region}")

try:
    # Note: Bucket configuration is required for all regions except us-east-1
    bucket_configuration = {"LocationConstraint": default_region}
    response = s3.create_bucket(Bucket=bucket_name, CreateBucketConfiguration=bucket_configuration)

    print("\n📦 AWS Response:")
    pp.pprint(response)

    if response["ResponseMetadata"]["HTTPStatusCode"] == 200:
        print(f"\n✅ Success! Bucket {bucket_name} created in {default_region}")
except Exception as e:
    print(f"❌ Error creating bucket: {str(e)}")

In [ ]:
# Upload our file to S3
# This shows how to transfer local files to your S3 bucket
print(f"⬆️  Uploading file to bucket: {bucket_name}")

try:
    s3.upload_file("my_content.txt", bucket_name, "my_content.txt")
    print("✅ Upload successful!")
    print(f"📍 File location: s3://{bucket_name}/my_content.txt")
    print(f"ℹ️ Note: The https URL https://{bucket_name}.s3.eu-west-1.amazonaws.com/my_content.txt")
    print("   won't work directly because S3 objects are private by default!")
    print("   We'll generate a pre-signed URL later to access this file via HTTPS.")

    # Verify the upload by listing objects in the bucket
    objects = s3.list_objects_v2(Bucket=bucket_name)
    print("\n📦 Current bucket contents:")
    for obj in objects.get("Contents", []):
        print(f"- {obj['Key']} ({obj['Size']} bytes)")
except Exception as e:
    print(f"❌ Error uploading file: {str(e)}")